# Digit Recognizer — *casa Zaccaria* notebook

Goal is **learning**, not chasing the leaderboard: the CNN dojo before real OCR/HTR.
We grow the pipeline version by version, each step with a **pre-declared success criterion**
and a logged **ADOPT / REJECT** verdict.

**House rules applied here**
- The out-of-fold (OOF) score is the *authoritative* signal. The public leaderboard is noise.
- Compare **like-for-like**: single-model OOF vs single-model OOF; same training regime on both sides.
- One variable per experiment, so we always know *which* change moved the needle.
- A pre-declared criterion is written *before* results; a FAIL is logged honestly, never re-rolled into a PASS.
- Don't tell stories about noise: on small validation splits, only movements beyond ±1–2 counts are signal.
- The experiment log lives inside this notebook; it is the write-up skeleton.

## 1. Setup

Reproducible seeds and a single config block. Set the Kaggle accelerator to **GPU**
(Settings → Accelerator) before the CNN sections.

In [ ]:
import os
import random
import numpy as np
import pandas as pd

SEED = 42
N_SPLITS = 5
DATA_DIR = "/kaggle/input/competitions/digit-recognizer"   # user-specified mount path

random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
print("Setup done. Seed:", SEED, "| DATA_DIR:", DATA_DIR)

## 2. Load data

785 columns: one `label` + 784 flattened pixels (28×28). Pixels 0–255 → scale to [0,1].
Scaling is not "cleaning"; it just helps the optimizer converge.

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

y = train["label"].values
X = train.drop(columns="label").values.astype("float32") / 255.0   # (n, 784)
X_test = test.values.astype("float32") / 255.0                     # (m, 784)
print("train:", X.shape, " test:", X_test.shape, " labels:", np.unique(y))

## 3. Quick EDA — *and the validation decision*

Two questions, both of which drive the validation scheme:

1. **Balanced classes?** If yes, plain accuracy is honest and no imbalance machinery is needed.
2. **Independent samples?** In *Trace the Ace* samples shared a `session_id`, forcing `GroupKFold`.
   Here each image is an independent digit — no grouping key — so the correct choice is
   `StratifiedKFold` (stratified on the label to keep each fold's class mix representative).

In [ ]:
import matplotlib.pyplot as plt

counts = pd.Series(y).value_counts().sort_index()
print(counts.to_dict())
print("min/max class share:", round(counts.min()/len(y), 4), "/", round(counts.max()/len(y), 4))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].bar(counts.index, counts.values)
ax[0].set_title("Class balance"); ax[0].set_xlabel("digit"); ax[0].set_ylabel("count")
ax[0].set_xticks(range(10))

grid = np.zeros((28, 28*10))
for d in range(10):
    grid[:, d*28:(d+1)*28] = X[y == d][0].reshape(28, 28)
ax[1].imshow(grid, cmap="gray_r"); ax[1].set_title("One sample per class"); ax[1].axis("off")
plt.tight_layout(); plt.show()

**Decision (logged):** classes ~balanced (~10% each) → accuracy is fair, no correction.
Samples independent → `StratifiedKFold(n_splits=5, shuffle=True)`. This *same split object* is
reused across every version so OOF scores are directly comparable version-to-version.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


def write_submission(pred, filename):
    sub = pd.DataFrame({"ImageId": np.arange(1, len(pred) + 1), "Label": pred})
    sub.to_csv(filename, index=False)
    print(f"{filename} written:", len(sub), "rows")
    return sub

## 4. v0 — logistic regression baseline

The floor. A linear model on raw pixels treats pixel (5,5) and (5,6) as unrelated columns —
no notion of *where* things are. Whatever it reaches is the ceiling of the
"an image is 784 scalars" mindset.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score

oof_v0 = cross_val_predict(
    LogisticRegression(max_iter=1000, C=0.1, n_jobs=-1),
    X, y, cv=skf, method="predict", n_jobs=-1,
)
cv_v0 = accuracy_score(y, oof_v0)
print("v0 OOF accuracy:", round(cv_v0, 4))   # recorded: 0.9207

clf = LogisticRegression(max_iter=1000, C=0.1, n_jobs=-1).fit(X, y)
_ = write_submission(clf.predict(X_test), "submission_v0.csv")

## 5. The CNN

Reshape the 784 columns into a 28×28×1 **image** and let convolutions learn features via
**locality** (small 3×3 filters look at one neighborhood) and **weight sharing** (the same filter
slides everywhere → translation tolerance, reinforced by `MaxPooling`).

`build_cnn` is defined **once** with three augmentation knobs (`translate`, `rotate`, `zoom`), all
defaulting to 0.0 → no augmentation. One definition serves every version, so architecture is provably
identical across experiments and augmentation is the *only* thing that ever changes. Augmentation is
active in training only (like Dropout); at inference it is a no-op, so every OOF/test prediction is on
**clean images**.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(SEED)

X_img = X.reshape(-1, 28, 28, 1)
X_test_img = X_test.reshape(-1, 28, 28, 1)


def build_cnn(translate=0.0, rotate=0.0, zoom=0.0):
    # Augmentation covers POSE (position, orientation, scale), not IDENTITY.
    # All-zero => bare CNN, identical architecture to every other version.
    aug = []
    if translate: aug.append(layers.RandomTranslation(translate, translate))
    if rotate:    aug.append(layers.RandomRotation(rotate))
    if zoom:      aug.append(layers.RandomZoom(zoom))

    inputs = keras.Input(shape=(28, 28, 1))
    x = keras.Sequential(aug, name="aug")(inputs) if aug else inputs
    x = layers.Conv2D(32, 3, activation="relu")(x)   # 32 filters slide over the image
    x = layers.MaxPooling2D()(x)                      # keep strongest response per region
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    m = keras.Model(inputs, outputs)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


build_cnn().summary()

### v1 — first CNN (no augmentation, EarlyStopping on `val_accuracy`)

Deliberately minimal so the v0 → v1 delta cleanly isolates *"the model now knows it is looking
at an image."*

**Pre-declared criterion:** beat v0's OOF (0.9207).  *Result: 0.9735 → PASS (+0.0528).*

In [ ]:
oof_v1 = np.zeros(len(y), dtype="int64")
test_probs_v1 = np.zeros((len(X_test_img), 10), dtype="float32")
early_acc = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                          restore_best_weights=True)

for fold, (tr, va) in enumerate(skf.split(X_img, y)):
    model = build_cnn()                                  # no augmentation
    model.fit(X_img[tr], y[tr], validation_data=(X_img[va], y[va]),
              epochs=20, batch_size=128, callbacks=[early_acc], verbose=2)
    oof_v1[va] = model.predict(X_img[va], verbose=0).argmax(1)      # single-model OOF (authoritative)
    test_probs_v1 += model.predict(X_test_img, verbose=0) / N_SPLITS  # 5-model bag (submission only)
    print(f"fold {fold}:", round(accuracy_score(y[va], oof_v1[va]), 4))

cv_v1 = accuracy_score(y, oof_v1)
print("\nv1 OOF accuracy:", round(cv_v1, 4), "| delta vs v0:", round(cv_v1 - cv_v0, 4),
      "|", "PASS" if cv_v1 > cv_v0 else "FAIL")
_ = write_submission(test_probs_v1.argmax(1), "submission_v1.csv")

> **OOF vs LB.** The OOF (0.9735 on our run) is a *single model per fold*; the submission is a
> *5-model bag*. So the LB (0.98025) sitting above the OOF is the bagging lift, not overfitting.
> Decisions use single-model OOF, compared like-for-like.

### v1 error analysis — measure, don't eyeball

A confusion matrix is **directional** (`i → j` ≠ `j → i`). Pair counts vary a little run-to-run,
so always compare *within* a run.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_v1 = confusion_matrix(y, oof_v1)

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ConfusionMatrixDisplay(cm_v1, display_labels=range(10)).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("v1 OOF confusion matrix"); plt.tight_layout(); plt.show()

off = cm_v1.copy(); np.fill_diagonal(off, 0)

pairs = sorted(((i, j, off[i, j]) for i in range(10) for j in range(10) if off[i, j] > 0),
               key=lambda t: t[2], reverse=True)
print("Top directional confusions (true -> pred):")
for i, j, n in pairs[:6]:
    print(f"  {i} -> {j}: {n:>3}  ({n / cm_v1[i].sum() * 100:.2f}% of the {i}s)")

sym = {(i, j): off[i, j] + off[j, i] for i in range(10) for j in range(i + 1, 10)}
print("\nTop confusable pairs (both directions):")
for (i, j), n in sorted(sym.items(), key=lambda t: t[1], reverse=True)[:5]:
    print(f"  {i} <-> {j}: {n:>3}  ({off[i, j]} one way, {off[j, i]} the other)")

## 6. v1b — controlled ablation: EarlyStopping on `val_loss`

**One variable changed vs v1:** monitor `val_loss` (smooth) instead of `val_accuracy` (a staircase
metric that plateaus and can stop training too early). Same architecture, no augmentation.

**Pre-declared criterion:** beat v1's OOF (0.9735).  *Result: 0.9811 → PASS (+0.0059). Adopted as
reference. A single model at 0.9811 beats v1's 5-model bag (LB 0.98025): "train better" before "bag more".*

In [ ]:
oof_v1b = np.zeros(len(y), dtype="int64")
early_loss = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                           restore_best_weights=True)

for fold, (tr, va) in enumerate(skf.split(X_img, y)):
    model = build_cnn()                                  # no augmentation, same architecture
    model.fit(X_img[tr], y[tr], validation_data=(X_img[va], y[va]),
              epochs=25, batch_size=128, callbacks=[early_loss], verbose=2)
    oof_v1b[va] = model.predict(X_img[va], verbose=0).argmax(1)
    print(f"fold {fold}:", round(accuracy_score(y[va], oof_v1b[va]), 4))

cv_v1b = accuracy_score(y, oof_v1b)
print("\nv1b OOF accuracy:", round(cv_v1b, 4), "| delta vs v1:", round(cv_v1b - cv_v1, 4),
      "|", "PASS" if cv_v1b > cv_v1 else "FAIL")

## 7. v2 — naive data augmentation (all three knobs at once)

Add pose variation (shift + rotate + zoom together) on the v1b base. Augmentation slows convergence,
so epochs/patience rise.

**Pre-declared criterion:** beat the adopted reference v1b (0.9811).
**Sub-hypothesis (logged before results):** augmentation should reduce 4↔9 and 8↔9 but leave 2↔7
~unchanged.

> **Result: 0.9704 → REJECT (−0.0107 vs v1b).** Kept as-run; the FAIL is diagnosed, not re-rolled.
> Fold epoch counts (19 / 5 / 5 / 6 / 6) revealed `restore_best_weights` reverting 4 of 5 folds to
> nearly untrained checkpoints — the augmented `val_loss` is noisy early and `patience=6` tripped. So
> the naive-aug result is **confounded by early stopping**, which is why Section 8 switches to a
> fixed-epoch, one-knob-at-a-time design.

In [ ]:
oof_v2 = np.zeros(len(y), dtype="int64")
test_probs_v2 = np.zeros((len(X_test_img), 10), dtype="float32")
early_v2 = keras.callbacks.EarlyStopping(monitor="val_loss", patience=6,
                                         restore_best_weights=True)

for fold, (tr, va) in enumerate(skf.split(X_img, y)):
    model = build_cnn(translate=0.10, rotate=0.04, zoom=0.10)   # all three knobs on
    hist = model.fit(X_img[tr], y[tr], validation_data=(X_img[va], y[va]),
                     epochs=40, batch_size=128, callbacks=[early_v2], verbose=2)
    oof_v2[va] = model.predict(X_img[va], verbose=0).argmax(1)  # clean images (aug off at inference)
    test_probs_v2 += model.predict(X_test_img, verbose=0) / N_SPLITS
    print(f"fold {fold}: acc={round(accuracy_score(y[va], oof_v2[va]), 4)} "
          f"| epochs_run={len(hist.history['loss'])}")   # watch the stop points!

cv_v2 = accuracy_score(y, oof_v2)
print("\nv2 OOF accuracy:", round(cv_v2, 4), "| delta vs v1b:", round(cv_v2 - cv_v1b, 4),
      "|", "PASS" if cv_v2 > cv_v1b else "FAIL")
_ = write_submission(test_probs_v2.argmax(1), "submission_v2.csv")

### v1 → v2: did augmentation move the pairs we predicted?

Compare confusion matrices, don't eyeball. Blue = fewer errors than v1.

In [ ]:
cm_v2 = confusion_matrix(y, oof_v2)

def pair(cm, i, j):
    return cm[i, j] + cm[j, i]

print("Pre-declared pairs (both directions):")
for i, j in [(2, 7), (7, 9), (4, 9), (8, 9)]:
    b, a = pair(cm_v1, i, j), pair(cm_v2, i, j)
    print(f"  {i}<->{j}: {b} -> {a}  ({a - b:+d})")

diff = (cm_v2 - cm_v1).astype(int); np.fill_diagonal(diff, 0)
m = max(1, int(np.abs(diff).max()))
fig, ax = plt.subplots(figsize=(5.4, 5.4))
im = ax.imshow(diff, cmap="RdBu_r", vmin=-m, vmax=m)
for i in range(10):
    for j in range(10):
        if diff[i, j]:
            ax.text(j, i, f"{diff[i, j]:+d}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Off-diagonal change: v2 - v1  (blue = fewer errors)")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## 8. Augmentation ablations — one knob at a time (diagnosis of the v2 FAIL)

The naive v2 mixed three knobs *and* an early-stopping confound, so its aggregate is uninformative.
Here we isolate each knob under a **fixed training regime**:

- **Fixed epochs, NO EarlyStopping** removes the data-dependent stop point (a hidden variable).
- **One knob per config**, each against the same no-augmentation reference in the same regime.

**SCREEN mode:** a single stratified split, not full 5-fold. A screening signal to rank the knobs
and watch the key pairs — *not* the authoritative OOF.

**Pre-declared hypothesis (logged before results):** translation helps or is neutral; zoom is mild;
rotation is the villain, expected to inflate **7↔9**.

In [ ]:
from sklearn.model_selection import train_test_split

SCREEN_EPOCHS = 25

configs = {
    "ref_noaug":   dict(),
    "a_translate": dict(translate=0.10),
    "b_rotate":    dict(rotate=0.04),
    "c_zoom":      dict(zoom=0.10),
}

Xtr, Xva, ytr, yva = train_test_split(X_img, y, test_size=0.2, stratify=y, random_state=SEED)
pair = lambda cm, i, j: cm[i, j] + cm[j, i]

results = {}
for name, cfg in configs.items():
    tf.random.set_seed(SEED)                    # same init + aug RNG start -> fair comparison
    model = build_cnn(**cfg)
    model.fit(Xtr, ytr, validation_data=(Xva, yva),
              epochs=SCREEN_EPOCHS, batch_size=128, verbose=0)
    pred = model.predict(Xva, verbose=0).argmax(1)
    cm = confusion_matrix(yva, pred)
    results[name] = dict(acc=accuracy_score(yva, pred),
                         p79=pair(cm, 7, 9), p49=pair(cm, 4, 9), p27=pair(cm, 2, 7))
    print(f"{name:12s} done  (val_acc={results[name]['acc']:.4f})")

ref = results["ref_noaug"]["acc"]
print("\n config        val_acc   d_acc     7<->9  4<->9  2<->7")
for name, r in results.items():
    print(f" {name:12s} {r['acc']:.4f}  {r['acc']-ref:+.4f}    {r['p79']:>4}   {r['p49']:>4}   {r['p27']:>4}")

### Ablation verdict (recorded)

| config | d_acc vs ref | 7↔9 | 4↔9 | 2↔7 | read |
|--------|:---:|:---:|:---:|:---:|------|
| ref_noaug (0.9886) | +0.0000 | 4 | 4 | 5 | reference |
| a_translate | **+0.0018** | 6 | 5 | 4 | helps |
| b_rotate | **−0.0020** | 6 | **18** | 5 | harms — wrecks 4↔9 |
| c_zoom | **+0.0013** | 3 | 5 | 5 | helps |

- **Only movement beyond noise is 4↔9 under rotation (4 → 18).** 7↔9 / 2↔7 wobble by ±1–2 = noise.
- **The pre-declared 7↔9 hypothesis was WRONG.** Rotation does not touch 7↔9; it destroys **4↔9**.
  Geometry, post-hoc: 7-vs-9 hinges on the *presence of a closed loop* (rotation-invariant), while
  4-vs-9 hinges on an *orientation-sensitive angular feature* that ±14° blurs.
- **Corollary — rotation is exonerated for v2's 7↔9 +49.** Clean rotation leaves 7↔9 flat, so that
  spike was the early-stopping confound, not augmentation. Isolating one variable at fixed epochs
  cleared the innocent knob.
- **Decision:** keep translation + zoom, drop rotation → carried into v3.

## 9. v3 — translation + zoom, rotation dropped (full 5-fold, fixed-epoch regime)

Two things changed vs v1b, so we control for both by running a matched pair on the same 5 folds:

- **v1c** — no augmentation, **fixed-epoch regime (no EarlyStopping)**. The screen showed fixed epochs
  train more fully than v1b's ES setup, but "changing the regime" is itself a variable — so v1c is the
  fair, re-based reference for this regime.
- **v3** — v1c **plus** translation + zoom. The *only* difference between v1c and v3 is augmentation,
  so `v3 − v1c` is the clean augmentation effect.

**Pre-declared criteria (written before results):** v3 must beat (1) **v1c** (isolates the augmentation
effect) **and** (2) the adopted historical best **v1b = 0.9811** (beats the best reference to date).

In [ ]:
FINAL_EPOCHS = 30

final_configs = {
    "v1c_noaug_fixed":   dict(),                           # fair reference in the fixed-epoch regime
    "v3_translate_zoom": dict(translate=0.10, zoom=0.10),  # one variable added: augmentation
}

final_oof, final_testprobs = {}, {}
for name, cfg in final_configs.items():
    oof = np.zeros(len(y), dtype="int64")
    tprobs = np.zeros((len(X_test_img), 10), dtype="float32")
    for fold, (tr, va) in enumerate(skf.split(X_img, y)):
        tf.random.set_seed(SEED + fold)
        model = build_cnn(**cfg)
        model.fit(X_img[tr], y[tr], epochs=FINAL_EPOCHS, batch_size=128, verbose=0)  # fixed, no ES
        oof[va] = model.predict(X_img[va], verbose=0).argmax(1)
        tprobs += model.predict(X_test_img, verbose=0) / N_SPLITS
    final_oof[name], final_testprobs[name] = oof, tprobs
    print(f"{name:18s} OOF: {accuracy_score(y, oof):.4f}")

cv_v1c = accuracy_score(y, final_oof["v1c_noaug_fixed"])
cv_v3 = accuracy_score(y, final_oof["v3_translate_zoom"])
print(f"\nv1c (fixed-epoch reference): {cv_v1c:.4f}")
print(f"v3  (translate + zoom):      {cv_v3:.4f}")
print(f"aug effect (v3 - v1c):       {cv_v3 - cv_v1c:+.4f}  |", "PASS" if cv_v3 > cv_v1c else "FAIL")
print(f"vs adopted best v1b (0.9811): {cv_v3 - cv_v1b:+.4f}  |", "PASS" if cv_v3 > cv_v1b else "FAIL")
_ = write_submission(final_testprobs["v3_translate_zoom"].argmax(1), "submission_v3.csv")

## 10. Experiment log  *(the write-up skeleton)*

Criterion written **before** results; verdict **after**. Rows marked *(screen)* are single-split
val_acc (ref_noaug 0.9886) and are **not** comparable to 5-fold OOF.

| Version | Description | CV (OOF acc) | Pre-declared criterion | Verdict |
|--------:|-------------|:------------:|------------------------|:-------:|
| v0  | Logistic regression on raw pixels | 0.9207 | baseline floor | REFERENCE |
| v1  | Minimal CNN, EarlyStopping on `val_accuracy` | 0.9735 | beat v0 (0.9207) | ADOPT (+0.0528) |
| v1b | v1 with EarlyStopping on `val_loss` | 0.9811 | beat v1 (0.9735) | **ADOPT (+0.0059)** |
| v2  | v1b base + naive augmentation (3 knobs) | 0.9704 | beat v1b (0.9811) | REJECT (−0.0107) |
| v2a | ablation: translation only *(screen)* | +0.0018 | rank knobs / watch pairs | translation helps |
| v2b | ablation: rotation only *(screen)* | −0.0020 | rank knobs / watch pairs | REJECT — harms 4↔9 |
| v2c | ablation: zoom only *(screen)* | +0.0013 | rank knobs / watch pairs | zoom helps |
| v1c | no-aug, fixed-epoch regime, 5-fold | *fill* | fair reference for the regime | *fill* |
| v3  | v1c + translation + zoom, 5-fold | *fill* | beat v1c **and** v1b (0.9811) | *fill* |
| v4  | v3 + BatchNorm / dropout / deeper head | — | beat v3 | *planned* |

**Running notes**
- v0: OOF 0.9207 ≈ LB 0.9205 → pipeline clean, OOF trustworthy.
- v1: OOF 0.9735 (single) vs LB 0.98025 (5-model bag). Gap = bagging lift, not overfitting.
- v1b: single model 0.9811 beats v1's 5-model bag → prioritise better training over more bagging.
- v2 REJECT. Fold epochs 19/5/5/6/6: early stopping reverted 4/5 folds to under-trained checkpoints;
  naive-aug result confounded. Diagnosis moved to fixed-epoch ablations.
- Ablation (fixed epochs, single split): translate +0.0018, zoom +0.0013, rotate −0.0020.
- **Rotation harms 4↔9 (4 → 18), NOT 7↔9** → pre-declared 7↔9 hypothesis was wrong; and v2's 7↔9 +49
  was the early-stopping confound, not rotation.
- 2↔7 stayed flat across every experiment → candidate for *irreducible* glyph ambiguity.

## 11. Next steps

- Read the v3 result: if `v3 > v1c` **and** `v3 > 0.9811`, adopt v3 as the new reference. If v3 beats
  v1c but not v1b, the fixed-epoch regime — not augmentation — was doing the work; log accordingly.
- **v4**: BatchNorm + dropout + a slightly deeper head; then light Optuna tuning with the current best
  enqueued as an anti-regression trial.
- Keep watching 2↔7: if it never moves, that is a *plateau* diagnosis (irreducible glyph ambiguity),
  not a modeling failure — and worth stating explicitly in the write-up.